In [ ]:
from pathlib import Path
from urllib.parse import urlencode, urlunsplit, parse_qs, urlparse

import re
import time
import io
import zipfile
import asyncio
import httpx


async def get_env(client):

    def extract_url(pattern, html):
        m = re.search(pattern, html)
        return m.group(1) if m else None

    response = await client.get(
        "https://3dencoder.com/SKP-to-blend",
        timeout=10
    )
    response.raise_for_status()

    html = response.text

    upload_url = extract_url(
        r'uploadURL\s*:\s*"([^"]+)"',
        html
    )

    upload_params = parse_qs(
        urlparse(upload_url).query
    )

    j = upload_params.get("j", [None])[0]
    cuid = upload_params.get("cuid", [None])[0]
    s = upload_params.get("s", [None])[0]

    file_path = Path("njc-b2.skp")
    target_path = Path(".blend/njc-b2/njc-b2.blend")

    file_size = file_path.stat().st_size

    def_params = {
        "j": j,
        "cuid": cuid,
        "s": s,
        "name": file_path.name,
        "size": file_size
    }

    return file_path, target_path, def_params


async def get_token(client, def_params={}):

    tk_params = {
        "Method": "tk"
    } | def_params

    token_url = urlunsplit((
        "https",
        "e1.3dwhere.com",
        "/eupload.ashx",
        urlencode(tk_params),
        ""
    ))

    print(f"TokenUrl: {token_url}\n")

    try:
        token_res = await client.get(token_url)
        token_res.raise_for_status()

        token_data = token_res.json()
        token = token_data.get("token")

    except Exception as e:
        print("Request for token failed.")
        print(e)
        return None

    print(f"Token: {token}")

    return token


async def get_fcode(client, file_path, token, def_params={}):

    upload_params = {
        "Method": "upload",
        "token": token,
        "client": "html5",
        "percen": 10
    } | def_params

    upload_url = urlunsplit((
        "https",
        "e1.3dwhere.com",
        "/eupload.ashx",
        urlencode(upload_params),
        ""
    ))

    print("Upload:", upload_url, "\n")

    try:
        with file_path.open("rb") as f:

            upload_res = await client.post(
                upload_url,
                content=f.read(),
                timeout=120
            )

        upload_res.raise_for_status()

        upload_data = upload_res.json()
        fcode = upload_data.get("message")

    except Exception as e:
        print("Failed to upload .skp ")
        print(repr(e))
        return None

    print(fcode)

    return fcode


async def get_zipurl(client, fcode):

    zipurl_params = {
        "action": "getinfo",
        "fcode": fcode
    }

    zipurl_url = urlunsplit((
        "https",
        "e1.3dwhere.com",
        "/json.aspx",
        urlencode(zipurl_params),
        ""
    ))

    zipfile_url = None

    print(f"Zipurl URL: {zipurl_url}")

    start = time.monotonic()

    while zipfile_url is None:

        try:
            zipurl_res = await client.get(
                zipurl_url,
                timeout=5
            )

            zipurl_res.raise_for_status()

            zipurl_data = zipurl_res.json()

            print(zipurl_data)

            zipfile_url = zipurl_data.get("zipfile")

            print(zipfile_url)

        except Exception as e:
            print("Request for zip url failed")
            print(e)

        if time.monotonic() - start > 120:
            break

        await asyncio.sleep(2)

    return zipfile_url


async def save_zipfile(client, zipfile_url, target_path):

    try:
        zip_res = await client.get(
            zipfile_url,
            timeout=120
        )

        zip_res.raise_for_status()

    except Exception as e:
        print("Request for zip file failed")
        print(e)
        return None

    print("Obtained Response for Zip File")

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    zip_path = target_path.with_suffix(".zip")

    zip_path.write_bytes(zip_res.content)

    with zipfile.ZipFile(
        io.BytesIO(zip_res.content)
    ) as z:

        file = z.namelist()[0]

        data = z.read(file)

        target_path.write_bytes(data)

    return target_path

In [9]:
from pathlib import Path
from urllib.parse import urlencode, urlunsplit, parse_qs, urlparse

import re
import time
import io
import zipfile
import asyncio
import httpx

class aobject(object):
    async def __new__(cls,*a,**kw):
        instance = super().__new__(cls)
        await instance.__init__(*a,**kw)
        return instance

    async def __init__(self):
        pass

class Session(aobject):
    async def __init__(self):
        async with httpx.AsyncClient() as client:
            
            response = await client.get(
                "https://3dencoder.com/SKP-to-blend",
                timeout=10
            )

            response.raise_for_status()
        
        html = response.text
    
        m = re.search(
            r'uploadURL\s*:\s*"([^"]+)"',
            html
        ).group(1)     
    
        upload_params = parse_qs(
            urlparse(m).query
        )
    
        j = upload_params.get("j", [None])[0]
        s = upload_params.get("s", [None])[0]
        cuid = upload_params.get("cuid", [None])[0]
        
        def_params = {
            "j": j,
            "cuid": cuid,
            "s": s,
        }
        
        self.def_params = def_params    
    

class ConvertSkp(aobject):
    async def __init__(self,filepath: Path):
        self.filepath = filepath
        self.filename = filepath.name
        self.filesize = filepath.stat().st_size

        self.savepath = Path("./.blend") / filepath.parent.relative_to(".skp") / filepath.name
        self.savepath.parent.mkdir(
            parents=True,
            exist_ok=True
        )



        self.file_params = {
            "name":filepath.name,
            "size":filepath.stat().st_size
        }


    async def get_env(self,client):
                
        response = await client.get(
            "https://3dencoder.com/SKP-to-blend",
            timeout=10
        )

        response.raise_for_status()
    
        html = response.text
    
        m = re.search(
            r'uploadURL\s*:\s*"([^"]+)"',
            html
        ).group(1)     
    
        upload_params = parse_qs(
            urlparse(m).query
        )
    
        j = upload_params.get("j", [None])[0]
        s = upload_params.get("s", [None])[0]
        cuid = upload_params.get("cuid", [None])[0]
        

        return s,j,cuid
    
    async def get_token(self,client:httpx.AsyncClient,s,cuid):
    
        tk_params = {
            "Method": "tk",
            "cuid": cuid,
            "s":s
        } | self.file_params
    
        token_url = urlunsplit((
            "https",
            "e1.3dwhere.com",
            "/eupload.ashx",
            urlencode(tk_params),
            ""
        ))
    
        print(f"TokenUrl: {token_url}\n")


        try:
            token_res = await client.get(token_url,timeout=50)
            token_res = token_res.json()
            token = token_res.get("token")

        except Exception as e:
            print("Request for token failed.")
            print(repr(e))
            return None
    
        print(f"Token: {token}")
    
        return token

    async def get_fcode(self,client,token,s,j,cuid):
    
        upload_params = {
            "Method": "upload",
            "token": token,
            "client": "html5",
            "percen": 10,
            "s":s,
            "j":j,
            "cuid":cuid
        } | self.file_params
    
        upload_url = urlunsplit((
            "https",
            "e1.3dwhere.com",
            "/eupload.ashx",
            urlencode(upload_params),
            ""
        ))
    
        print("Upload:", upload_url, "\n")
        # while fcode is None:
        #     try:
        #         with self.filepath.open("rb") as f:
        
        #             upload_res = await client.post(
        #                 upload_url,
        #                 content=f.read(),
        #                 timeout=5
        #             )
        
        #         upload_res.raise_for_status()
        
        #         upload_data = upload_res.json()
        #         fcode = upload_data.get("message")
        
        #     except Exception as e:
        #         print("Failed to upload .skp ")
        #         print(repr(e))

        #     if time.monotonic() - start > 120:
        #         return None

        #     await asyncio.sleep(10)

        try:
            with self.filepath.open("rb") as f:
    
                upload_res = await client.post(
                    upload_url,
                    content=f.read(),
                    timeout=240
                )
    
            upload_res.raise_for_status()
    
            upload_data = upload_res.json()
            fcode = upload_data.get("message")
    
        except Exception as e:
            print("Failed to upload .skp ")
            print(repr(e))
            return None
        
    
        return fcode

    async def get_zipurl(self,client,fcode):
    
        zipurl_params = {
            "action": "getinfo",
            "fcode": fcode
        }
    
        zipurl_url = urlunsplit((
            "https",
            "e1.3dwhere.com",
            "/json.aspx",
            urlencode(zipurl_params),
            ""
        ))
    
        zipurl = None
    
        print(f"Zipurl URL: {zipurl_url}")
    
        start = time.monotonic()
    
        while zipurl is None:
    
            try:
                zipurl_res = await client.get(
                    zipurl_url,
                    timeout=10
                )
    
                zipurl_res.raise_for_status()
    
                zipurl_data = zipurl_res.json()
    
                print(zipurl_data)
    
                zipurl = zipurl_data.get("zipfile")
    
                print(f"Zip URL: {zipurl}")
    
            except Exception as e:
                print("Request for zip url failed")
                print(e)
    
            if time.monotonic() - start > 120:
                break
    
            await asyncio.sleep(2)
    
        return zipurl
    
    async def save_zipfile(self,client,zipurl):
    
        try:
            zip_res = await client.get(
                zipurl,
                timeout=240
            )
    
            zip_res.raise_for_status()
    
        except Exception as e:
            print("Request for zip file failed")
            print(repr(e))
            return None
    
        print("Obtained Response for Zip File")
    

    
        zip_path = self.savepath.with_suffix(".zip")
        blend_path = self.savepath.with_suffix(".blend")

        for file in [zip_path,blend_path]:
            if file.is_file():
                file.unlink()

    
        zip_path.write_bytes(zip_res.content)
    
        with zipfile.ZipFile(
            io.BytesIO(zip_res.content)
        ) as z:
    
            file = z.namelist()[0]
    
            data = z.read(file)
    
            blend_path.write_bytes(data)

        return True
    

    @staticmethod
    async def first_success(tasks):
        pending = set(tasks)
    
        while pending:
            done, pending = await asyncio.wait(
                pending,
                return_when=asyncio.FIRST_COMPLETED
            )
    
            for task in done:
                try:
                    result = task.result()
                except Exception as e:
                    print(f"Attempt failed: {repr(e)}")
                    continue
    
                if result:
                    # We got our zip URL.
                    # Cancel every other conversion.
                    for other in pending:
                        other.cancel()
    
                    await asyncio.gather(
                        *pending,
                        return_exceptions=True
                    )
    
                    return result
    
        return None

    async def _convert(self, client):
        s, j, cuid = await self.get_env(client)

        token = await self.get_token(client, s, cuid)

        if token is None:
            return None

        fcode = await self.get_fcode(
            client,
            token,
            s,
            j,
            cuid
        )

        if fcode is None:
            return None

        zipurl = await self.get_zipurl(
            client,
            fcode
        )

        if zipurl is None:
            return None

        # Stop the pipeline here.
        # The zip URL is the "success" result.
        return zipurl
            
        

    async def convert(self):
        async with httpx.AsyncClient() as client:
            convert_tasks = [
                asyncio.create_task(
                    self._convert(client)
                )
                for _ in range(4)
            ]
    
            zipurl = await self.first_success(convert_tasks)
    
            if zipurl is None:
                return False
    
            print(f"Winning zip URL: {zipurl}")
    
            return await self.save_zipfile(
                client,
                zipurl
            )

    
model = "b3"
instance = await ConvertSkp(Path(f".skp/njc-{model}/njc-{model}.skp"))
print(repr(instance.savepath))
await instance.convert()

WindowsPath('.blend/njc-b3/njc-b3.skp')
TokenUrl: https://e1.3dwhere.com/eupload.ashx?Method=tk&cuid=461048b9b0aa8dd07c739e9883c0339e&s=9d963cc50b815419f1d32e200c1febca&name=njc-b3.skp&size=5816149

TokenUrl: https://e1.3dwhere.com/eupload.ashx?Method=tk&cuid=60677bd93ed8d87aa5ca5c6cc95bd21f&s=0f6ed9bb333d1943a77b18fc61cb50f9&name=njc-b3.skp&size=5816149

TokenUrl: https://e1.3dwhere.com/eupload.ashx?Method=tk&cuid=ca112383fc4aa408275c8b74905c3e35&s=4d35913c310224fb2abb12432750939b&name=njc-b3.skp&size=5816149

TokenUrl: https://e1.3dwhere.com/eupload.ashx?Method=tk&cuid=7f770843fa93cb863c843b5664d84b4e&s=972c47cbc8ae7827dbace7c74ebbc8d3&name=njc-b3.skp&size=5816149

Token: F1B792E07C8F44C7727B68923624FDDE
Upload: https://e1.3dwhere.com/eupload.ashx?Method=upload&token=F1B792E07C8F44C7727B68923624FDDE&client=html5&percen=10&s=0f6ed9bb333d1943a77b18fc61cb50f9&j=BF4C2F3D8B8ECB6306D76082A90FDF4810E7A24E291D3B55503527DAEBE5BC6B45CBB9DFC2D98FA7AB6355F1BB0C978D4EC65FAE21227603026C3D0C309A674

True

In [8]:
from pathlib import Path
from urllib.parse import urlencode, urlunsplit, parse_qs, urlparse

import re
import time
import io
import zipfile
import requests


def get_env(session):
    def extract_url(pattern, html):
        m = re.search(pattern, html)
        return m.group(1) if m else None

    response = session.get(
        "https://3dencoder.com/model-to-blend",
        timeout=10
    )
    response.raise_for_status()

    html = response.text

    upload_url = extract_url(
        r'uploadURL\s*:\s*"([^"]+)"',
        html
    )

    if upload_url is None:
        raise RuntimeError("Could not find uploadURL")

    upload_params = parse_qs(
        urlparse(upload_url).query
    )

    j = upload_params.get("j", [None])[0]
    cuid = upload_params.get("cuid", [None])[0]
    s = upload_params.get("s", [None])[0]

    file_path = Path(".skp/njc-b1/njc-b1.skp")
    target_path = Path(".blend/njc-b1/njc-b1.blend")

    file_size = file_path.stat().st_size

    def_params = {
        "j": j,
        "cuid": cuid,
        "s": s,
        "name": file_path.name,
        "size": file_size
    }

    return file_path, target_path, def_params


def get_token(session, def_params=None):
    if def_params is None:
        def_params = {}

    tk_params = {
        "Method": "tk"
    } | def_params

    token_url = urlunsplit((
        "https",
        "e1.3dwhere.com",
        "/eupload.ashx",
        urlencode(tk_params),
        ""
    ))

    print(f"TokenUrl: {token_url}\n")

    try:
        token_res = session.get(
            token_url,
            timeout=30
        )

        token_res.raise_for_status()

        token_data = token_res.json()
        token = token_data.get("token")

    except Exception as e:
        print("Request for token failed.")
        print(repr(e))
        return None

    print(f"Token: {token}")

    return token


def get_fcode(session, file_path, token, def_params=None):
    if def_params is None:
        def_params = {}

    upload_params = {
        "Method": "upload",
        "token": token,
        "client": "html5",
        "percen": 10
    } | def_params

    upload_url = urlunsplit((
        "https",
        "e1.3dwhere.com",
        "/eupload.ashx",
        urlencode(upload_params),
        ""
    ))

    print("Upload:", upload_url, "\n")

    try:
        with file_path.open("rb") as f:
            data = f.read()

        upload_res = session.post(
            upload_url,
            data=data,
            timeout=120
        )

        upload_res.raise_for_status()

        upload_data = upload_res.json()
        fcode = upload_data.get("message")

    except Exception as e:
        print("Failed to upload .skp")
        print(repr(e))
        return None

    print(fcode)

    return fcode


def get_zipurl(session, fcode):
    zipurl_params = {
        "action": "getinfo",
        "fcode": fcode
    }

    zipurl_url = urlunsplit((
        "https",
        "e1.3dwhere.com",
        "/json.aspx",
        urlencode(zipurl_params),
        ""
    ))

    zipfile_url = None

    print(f"Zipurl URL: {zipurl_url}")

    start = time.monotonic()

    while zipfile_url is None:

        try:
            zipurl_res = session.get(
                zipurl_url,
                timeout=5
            )

            zipurl_res.raise_for_status()

            zipurl_data = zipurl_res.json()

            print(zipurl_data)

            zipfile_url = zipurl_data.get("zipfile")

            print(zipfile_url)

        except Exception as e:
            print("Request for zip url failed")
            print(repr(e))

        if time.monotonic() - start > 120:
            break

        time.sleep(2)

    return zipfile_url


def save_zipfile(session, zipfile_url, target_path):
    if zipfile_url is None:
        return None

    try:
        zip_res = session.get(
            zipfile_url,
            timeout=120
        )

        zip_res.raise_for_status()

    except Exception as e:
        print("Request for zip file failed")
        print(repr(e))
        return None

    print("Obtained Response for Zip File")

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    zip_path = target_path.with_suffix(".zip")

    zip_path.write_bytes(zip_res.content)

    with zipfile.ZipFile(
        io.BytesIO(zip_res.content)
    ) as z:

        file = z.namelist()[0]

        data = z.read(file)

        target_path.write_bytes(data)

    return target_path



with requests.Session() as session:

    file_path, target_path, def_params = get_env(session)

    token = get_token(
        session,
        def_params
    )

    if token is None:
        raise RuntimeError("Failed to obtain token")

    fcode = get_fcode(
        session,
        file_path,
        token,
        def_params
    )

    if fcode is None:
        raise RuntimeError("Failed to obtain fcode")

    zipfile_url = get_zipurl(
        session,
        fcode
    )

    if zipfile_url is None:
        raise RuntimeError("Timed out waiting for ZIP URL")

    target = save_zipfile(
        session,
        zipfile_url,
        target_path
    )

    print(f"Saved: {target}")

TokenUrl: https://e1.3dwhere.com/eupload.ashx?Method=tk&j=BF4C2F3D8B8ECB6306D76082A90FDF4810E7A24E291D3B55503527DAEBE5BC6B45CBB9DFC2D98FA7AB6355F1BB0C978D4EC65FAE21227603026C3D0C309A67469353F63FE912E48D80355D65C3F9D5E9373E5473B2664684286F73FCE2C43870&cuid=05da2ee6ff6e5d749080a237ba264d34&s=681a8c6650bf780ed8bca682613d4155&name=njc-b1.skp&size=1665835

Token: A64451F43DB6D4FBDC6C7CFE525A06B6
Upload: https://e1.3dwhere.com/eupload.ashx?Method=upload&token=A64451F43DB6D4FBDC6C7CFE525A06B6&client=html5&percen=10&j=BF4C2F3D8B8ECB6306D76082A90FDF4810E7A24E291D3B55503527DAEBE5BC6B45CBB9DFC2D98FA7AB6355F1BB0C978D4EC65FAE21227603026C3D0C309A67469353F63FE912E48D80355D65C3F9D5E9373E5473B2664684286F73FCE2C43870&cuid=05da2ee6ff6e5d749080a237ba264d34&s=681a8c6650bf780ed8bca682613d4155&name=njc-b1.skp&size=1665835 

sox1d9iq55bp541m
Zipurl URL: https://e1.3dwhere.com/json.aspx?action=getinfo&fcode=sox1d9iq55bp541m
{'stat': 0, 'errcount': 0, 'waitcount': 1, 'msg': ''}
None
{'stat': 2, 'msg': ''}
None
